# Train Swedish Train Delay Model (Trafikverket)

This notebook loads data from **`regression.db`** (table `delays_regression`), trains a regression model to predict arrival delay (in minutes), evaluates it, and exports:

- `delay_model.joblib` — the trained model
- `model_card.md` — short report on metrics & features

It also includes a **backfill & write** section showing how to insert predictions into a production SQLite DB table (e.g., `train_data.db`).

In [1]:
# --- Setup
import os, sqlite3, math, json
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# ML
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
import joblib

DB_PATH = os.environ.get('REGRESSION_DB', 'regression.db')
assert os.path.exists(DB_PATH), f"Database not found: {DB_PATH}"
DB_PATH

'regression.db'

In [ ]:
# Vi laddar in all data som vi behöver i en dataframe
# En rad i delays_regression är en planerad ankomst av ett specifikt tåg på en specifik plats

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql(
    """
    SELECT 
      journey_id, train_id, journey_date,
      station_signature, station_name, lon, lat,
      advertised_time, activity_type,
      delay_minutes, is_canceled,
      hour, weekday, prev_delay_minutes
    FROM delays_regression
    WHERE delay_minutes IS NOT NULL
    """,
    conn
)
conn.close()
print(df.shape)
df.head(3)

(92297, 14)


,journey_id,train_id,journey_date,station_signature,station_name,lon,lat,advertised_time,activity_type,delay_minutes,is_canceled,hour,weekday,prev_delay_minutes
0,3596,1,2025-10-22,Nr,Norrköping C,16.181679,58.596266,2025-10-22T00:56:00Z,Ankomst,2,0,0,2,NaN
1,3596,1,2025-10-22,Lp,Linköping C,15.624342,58.417037,2025-10-22T01:27:00Z,Ankomst,5,0,1,2,2.0
2,3596,1,2025-10-22,Lp,Linköping C,15.624342,58.417037,2025-10-22T01:27:00Z,Ankomst,5,0,1,2,5.0


In [ ]:
# vi kör fillna på första avgången för att inte ha massa NaNs där
df['is_canceled'] = df['is_canceled'].astype(int)
df['prev_delay_minutes'] = df['prev_delay_minutes'].fillna(0)

# Clippar bort extrema outliers, egentligen kanske man ska kika på dom först med boxplots eller nåt men skippar det just nu
df['delay_minutes'] = df['delay_minutes'].clip(lower=-20, upper=300)


# Vi väljer vilka features som vi tror påverkar hur sen vi blir.
# den starkaste indikatorn här är ju deffinitivt prev_delay
# borde lägga till distance to next stop och operator här sen
features = [
    'hour','weekday','prev_delay_minutes',
    'lon','lat','is_canceled',
    'station_signature'
]

# vad vi vill räkna ut
target = 'delay_minutes'

X = df[features].copy()
y = df[target].astype(float).values

# delar upp det i numerisk och kategorisk data
numeric_cols = ['hour','weekday','prev_delay_minutes','lon','lat','is_canceled']
cat_cols = ['station_signature']

pre = ColumnTransformer(
    transformers=[
        ('num','passthrough', numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# här väljer vi vilken regressionsmodel vi vill använda. Den här är duktig på massa olika interaktioner, t.ex stations påverkan på försening
model = HistGradientBoostingRegressor(
    max_depth=None, learning_rate=0.08, max_iter=400,
    l2_regularization=0.0, min_samples_leaf=20,
    random_state=42
)

pipe = Pipeline([
    ('pre', pre),
    ('model', model)
])

# delar upp det i 20% train split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)

pred = pipe.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
print({'MAE_min': round(mae,2), 'R2': round(r2,3), 'n_train': len(X_train), 'n_test': len(X_test)})

{'MAE_min': 1.02, 'R2': 0.793, 'n_train': 73837, 'n_test': 18460}


In [ ]:
# --- Persist artifacts
ARTIFACT_DIR = 'artifacts'
os.makedirs(ARTIFACT_DIR, exist_ok=True)
MODEL_PATH = os.path.join(ARTIFACT_DIR, 'delay_model.joblib')
PREPROC_PATH = os.path.join(ARTIFACT_DIR, 'delay_preprocessor.joblib')

# Sparar hela pipelinen i en fil för att kunna återanvända modellen senare
FULL_PIPELINE_PATH = os.path.join(ARTIFACT_DIR, 'delay_pipeline.joblib')
joblib.dump(pipe, FULL_PIPELINE_PATH)

# Skapar en liten md fil för att se hur bra modellen är
with open(os.path.join(ARTIFACT_DIR, 'model_card.md'), 'w') as f:
    f.write(f"""
# Delay Prediction Model — Model Card

- Trained: {datetime.utcnow().isoformat()}Z
- Algorithm: sklearn HistGradientBoostingRegressor
- Features: {features}
- Target: delay_minutes (clipped to [-20, 300])
- Split: 80/20 train/test

## Metrics (test)
- MAE (minutes): {mae:.2f}
- R^2: {r2:.3f}

## Notes
- `prev_delay_minutes` approximates delay propagation.
- `station_signature` is one-hot encoded; lon/lat included for spatial signal.
    """)

FULL_PIPELINE_PATH, os.path.join(ARTIFACT_DIR, 'model_card.md')

# det som händer när vi kallar på modellen i våran backend senare är
# vi skickar in en dataframe med tid, dag, försening, kordinater, is_canceled och station
# baserat på det så räknar modellen ut hur sen vi borde bli till nästa station


/tmp/ipykernel_14583/3417784812.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  - Trained: {datetime.utcnow().isoformat()}Z


('artifacts/delay_pipeline.joblib', 'artifacts/model_card.md')